# MV-Adapter: distribution-preserving view-noise experiments

This notebook supports two strictly isolated profiles:

- `QUICK_SMOKE` (default): one input, one seed, and two strengths. It is only a code-path smoke test and must not be reported as a formal experiment.
- `FULL`: exactly five user-provided inputs, three seeds, and four strengths.

Every run directory is derived from the checked-out code commit and a hash of the complete experiment configuration. Legacy Sobol, low-pass, and callback outputs remain only in `mvadapter_nile_experiments_colab.ipynb` as failure-analysis artifacts.


## 1. Mount Google Drive

Checkpoints, user inputs, immutable experiment configuration, manifests, diagnostics, and generated artifacts are persisted on Drive. Checkpoint download can use a CPU runtime; SDXL inference requires a GPU runtime.


In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "This kernel is not Google Colab. Select a Colab kernel in VS Code."
    ) from exc

drive.mount("/content/drive")


## 2. Select the experiment profile and immutable parameters

Keep `RUN_PROFILE="QUICK_SMOKE"` while checking installation and command wiring. Change it to `"FULL"` only after placing at least five real, asymmetric inputs in `USER_INPUT_DIR`.


In [ ]:
import hashlib
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

GITHUB_REPO = "https://github.com/WANG-Ruipeng/MV-Adapter.git"
BRANCH = "main"
COLAB_REPO = Path("/content/MV-Adapter")
MODEL_ROOT = Path("/content/drive/MyDrive/ModelWeights/MV-Adapter")
PROJECT_ROOT = Path("/content/drive/MyDrive/Colab_Projects/MV-Adapter")
EXPERIMENTS_ROOT = PROJECT_ROOT / "nile_distribution_preserving_experiments"
USER_INPUT_DIR = PROJECT_ROOT / "nile_distribution_preserving_inputs"
ADAPTER_DIR = MODEL_ROOT / "mv-adapter"
HF_HOME = MODEL_ROOT / "huggingface"

BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
BASE_MODEL_REQUESTED_REVISION = "main"
VAE_MODEL = "madebyollin/sdxl-vae-fp16-fix"
VAE_MODEL_REQUESTED_REVISION = "main"
ADAPTER_REPO = "huanngzh/mv-adapter"
ADAPTER_REQUESTED_REVISION = "main"
ADAPTER_WEIGHT = "mvadapter_i2mv_sdxl.safetensors"
BIREFNET_MODEL = "ZhengPeng7/BiRefNet"
BIREFNET_REQUESTED_REVISION = "main"

TEXT_PROMPT = "high quality, detailed object"
AZIMUTHS = [0, 45, 90, 180, 270, 315]
METHODS = [
    "iid_default",
    "iid_external",
    "shared_full",
    "spectral_global_corr",
    "camera_rbf_corr",
    "nested_tree_a",
    "nested_tree_ab",
]
FORBIDDEN_METHODS = {
    "flat_sobol", "lowpass_shared", "nile_v", "nile_vt", "nile_vtp"
}
FREQUENCY_SCALE = 0.12
CAMERA_LENGTH_SCALE = 0.8
NUM_INFERENCE_STEPS = 30
GUIDANCE_SCALE = 3.0
REMOVE_BACKGROUND = True

RUN_PROFILE = "QUICK_SMOKE"  # Change to "FULL" for the formal 5-input run.
assert RUN_PROFILE in {"QUICK_SMOKE", "FULL"}
IS_FULL = RUN_PROFILE == "FULL"
SEEDS = [0, 1, 2] if IS_FULL else [0]
STRENGTHS = [0.15, 0.30, 0.45, 0.60] if IS_FULL else [0.15, 0.30]

assert not (set(METHODS) & FORBIDDEN_METHODS)
assert METHODS[:3] == ["iid_default", "iid_external", "shared_full"]
if IS_FULL:
    assert SEEDS == [0, 1, 2]
    assert STRENGTHS == [0.15, 0.30, 0.45, 0.60]

for path in (ADAPTER_DIR, HF_HOME, EXPERIMENTS_ROOT, USER_INPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["PYTHONUNBUFFERED"] = "1"

print("profile:", RUN_PROFILE)
if not IS_FULL:
    print("QUICK_SMOKE is only a smoke test; do not use it as a formal result.")
print("user input directory:", USER_INPUT_DIR)


## 3. Download checkpoints to Drive (CPU is sufficient)

Requested Hugging Face branches are resolved to immutable commit SHAs before download. Both requested and resolved revisions are stored in the experiment configuration and grid records. A gated model can read `HF_TOKEN` from Colab Secrets without printing it.


In [ ]:
try:
    from huggingface_hub import hf_hub_download, model_info, snapshot_download
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"],
        check=True,
    )
    from huggingface_hub import hf_hub_download, model_info, snapshot_download

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

def resolve_hf_revision(repo_id, requested_revision):
    info = model_info(
        repo_id=repo_id,
        revision=requested_revision,
        token=HF_TOKEN or None,
    )
    resolved = info.sha
    assert (
        isinstance(resolved, str)
        and len(resolved) == 40
        and all(character in "0123456789abcdef" for character in resolved.lower())
    ), f"Could not resolve an immutable commit for {repo_id}@{requested_revision}"
    print("resolved:", repo_id, requested_revision, "->", resolved)
    return resolved

BASE_MODEL_REVISION = resolve_hf_revision(
    BASE_MODEL, BASE_MODEL_REQUESTED_REVISION
)
VAE_MODEL_REVISION = resolve_hf_revision(
    VAE_MODEL, VAE_MODEL_REQUESTED_REVISION
)
ADAPTER_REVISION = resolve_hf_revision(
    ADAPTER_REPO, ADAPTER_REQUESTED_REVISION
)
BIREFNET_REVISION = (
    resolve_hf_revision(BIREFNET_MODEL, BIREFNET_REQUESTED_REVISION)
    if REMOVE_BACKGROUND
    else None
)

adapter_file = hf_hub_download(
    repo_id=ADAPTER_REPO,
    revision=ADAPTER_REVISION,
    filename=ADAPTER_WEIGHT,
    local_dir=ADAPTER_DIR,
    token=HF_TOKEN or None,
)
print("adapter:", adapter_file)

PRELOAD_MODEL_SNAPSHOTS = True
model_snapshots = [
    (BASE_MODEL, BASE_MODEL_REVISION),
    (VAE_MODEL, VAE_MODEL_REVISION),
]
if REMOVE_BACKGROUND:
    model_snapshots.append((BIREFNET_MODEL, BIREFNET_REVISION))

if PRELOAD_MODEL_SNAPSHOTS:
    for repo_id, revision in model_snapshots:
        print("preloading:", repo_id, "revision:", revision)
        snapshot_download(
            repo_id=repo_id,
            revision=revision,
            token=HF_TOKEN or None,
        )
else:
    print("Base model and VAE will be cached on Drive during first inference.")


## 4. Clone an exact code revision and install dependencies

Push the current sampler, diagnostics, grid, and evaluator code to GitHub before running this notebook. The checked-out commit becomes part of the experiment identity.


In [ ]:
def run_streaming(command, *, cwd=None, check=True):
    command = [str(item) for item in command]
    print("\n$", shlex.join(command), flush=True)
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        check=False,
    )
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, command)
    return result.returncode

if (COLAB_REPO / ".git").is_dir():
    run_streaming(["git", "-C", COLAB_REPO, "fetch", "origin", BRANCH])
    run_streaming(["git", "-C", COLAB_REPO, "checkout", BRANCH])
    run_streaming(
        ["git", "-C", COLAB_REPO, "pull", "--ff-only", "origin", BRANCH]
    )
else:
    run_streaming(
        ["git", "clone", "--branch", BRANCH, GITHUB_REPO, COLAB_REPO]
    )

required_files = [
    "scripts/diagnose_nile_latents.py",
    "scripts/inference_i2mv_sdxl_nile.py",
    "scripts/run_nile_grid.py",
    "scripts/eval_multiview_consistency.py",
    "mvadapter/nile/spectral_gaussian.py",
    "mvadapter/nile/nested_elements.py",
]
missing = [name for name in required_files if not (COLAB_REPO / name).is_file()]
assert not missing, f"Checked-out branch is missing experiment files: {missing}"

unversioned_required = []
for name in required_files:
    result = subprocess.run(
        [
            "git", "-C", COLAB_REPO, "ls-files",
            "--error-unmatch", "--", name,
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    )
    if result.returncode != 0:
        unversioned_required.append(name)
assert not unversioned_required, (
    f"Experiment files are not tracked by git: {unversioned_required}"
)

repository_changes = subprocess.check_output(
    [
        "git", "-C", COLAB_REPO, "status", "--porcelain",
        "--untracked-files=all",
    ],
    text=True,
).strip()
assert not repository_changes, (
    "Colab repository has tracked or untracked files. Commit/push them or use a fresh clone."
)
GIT_COMMIT = subprocess.check_output(
    ["git", "-C", COLAB_REPO, "rev-parse", "HEAD"], text=True
).strip()
assert len(GIT_COMMIT) == 40
print("code revision:", GIT_COMMIT)


In [ ]:
def run_pip(*args):
    run_streaming([sys.executable, "-m", "pip", *args])

run_pip("install", "-r", str(COLAB_REPO / "requirements-colab.txt"))
run_pip(
    "install",
    "--no-build-isolation",
    "git+https://github.com/NVlabs/nvdiffrast.git",
)
run_pip("install", "-e", str(COLAB_REPO), "--no-deps")
print("Dependencies installed.")


## 5. Validate inputs, GPU, and create an isolated experiment root

`FULL` never uses the demo penguin: it hard-fails unless at least five user files exist, then uses exactly five. `QUICK_SMOKE` may fall back to the repository penguin only to verify that the pipeline runs.

The experiment ID is `<profile>-<commit>-<complete-config-hash>`. Therefore QUICK/FULL, different inputs, model revisions, or any generation setting cannot share a root or manifest.


In [ ]:
import torch

assert torch.cuda.is_available(), "No CUDA device; switch to a Colab GPU runtime."
run_streaming(["nvidia-smi"])
print("GPU:", torch.cuda.get_device_name(0))

supported = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
user_images = sorted(
    path.resolve()
    for path in USER_INPUT_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in supported
)
USING_FALLBACK = False
if IS_FULL:
    assert len(user_images) >= 5, (
        "FULL requires at least five user-provided images in "
        f"{USER_INPUT_DIR}; the demo penguin is forbidden."
    )
    INPUT_IMAGES = user_images[:5]
    assert len(INPUT_IMAGES) == 5
else:
    if user_images:
        INPUT_IMAGES = user_images[:1]
    else:
        fallback = (
            COLAB_REPO
            / "assets/demo/i2mv/A_juvenile_emperor_penguin_chick.png"
        ).resolve()
        assert fallback.is_file()
        INPUT_IMAGES = [fallback]
        USING_FALLBACK = True
        print("Using the demo penguin for QUICK_SMOKE only.")

assert INPUT_IMAGES
assert not (IS_FULL and USING_FALLBACK)
if IS_FULL:
    assert len(INPUT_IMAGES) == 5
else:
    assert len(INPUT_IMAGES) == 1

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

ADAPTER_SHA256 = sha256_file(adapter_file)

input_records = [
    {"path": str(path), "sha256": sha256_file(path)}
    for path in INPUT_IMAGES
]
EXPERIMENT_CONFIG = {
    "run_profile": RUN_PROFILE,
    "code_revision": GIT_COMMIT,
    "models": {
        "base_model": {
            "identifier": BASE_MODEL,
            "requested_revision": BASE_MODEL_REQUESTED_REVISION,
            "revision": BASE_MODEL_REVISION,
        },
        "vae_model": {
            "identifier": VAE_MODEL,
            "requested_revision": VAE_MODEL_REQUESTED_REVISION,
            "revision": VAE_MODEL_REVISION,
        },
        "adapter": {
            "identifier": ADAPTER_REPO,
            "requested_revision": ADAPTER_REQUESTED_REVISION,
            "revision": ADAPTER_REVISION,
            "weight": ADAPTER_WEIGHT,
            "sha256": ADAPTER_SHA256,
        },
        "birefnet": {
            "identifier": BIREFNET_MODEL,
            "requested_revision": BIREFNET_REQUESTED_REVISION,
            "revision": BIREFNET_REVISION,
        },
    },
    "inputs": input_records,
    "using_fallback": USING_FALLBACK,
    "methods": METHODS,
    "seeds": SEEDS,
    "strengths": STRENGTHS,
    "text_prompt": TEXT_PROMPT,
    "azimuth_deg": AZIMUTHS,
    "frequency_scale": FREQUENCY_SCALE,
    "camera_length_scale": CAMERA_LENGTH_SCALE,
    "num_inference_steps": NUM_INFERENCE_STEPS,
    "guidance_scale": GUIDANCE_SCALE,
    "remove_background": REMOVE_BACKGROUND,
}
canonical_config = json.dumps(
    EXPERIMENT_CONFIG,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)
CONFIG_HASH = hashlib.sha256(canonical_config.encode("utf-8")).hexdigest()
EXPERIMENT_ID = (
    f"{RUN_PROFILE.lower()}-{GIT_COMMIT[:12]}-{CONFIG_HASH[:16]}"
)
EXPERIMENT_ROOT = EXPERIMENTS_ROOT / EXPERIMENT_ID
OUTPUT_ROOT = EXPERIMENT_ROOT / "runs"
PLAN_MANIFEST = EXPERIMENT_ROOT / "plan.json"
MANIFEST_FILE = EXPERIMENT_ROOT / "manifest.json"
DIAGNOSTICS_FILE = EXPERIMENT_ROOT / "latent_diagnostics.json"
LIGHTWEIGHT_FILE = EXPERIMENT_ROOT / "collapse_diagnostics.json"
MET3R_FILE = EXPERIMENT_ROOT / "met3r_and_guardrails.json"
CONFIG_FILE = EXPERIMENT_ROOT / "notebook_config.json"
for path in (EXPERIMENT_ROOT, OUTPUT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

CONFIG_FILE.write_text(
    json.dumps(
        {
            "schema_version": 1,
            "experiment_id": EXPERIMENT_ID,
            "config_hash": CONFIG_HASH,
            "configuration": EXPERIMENT_CONFIG,
        },
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)
print("experiment ID:", EXPERIMENT_ID)
print("experiment root:", EXPERIMENT_ROOT)
for item in input_records:
    print(" -", item["path"], item["sha256"][:12])


## 6. Mandatory latent-distribution gate

This runs before loading SDXL. Any failure in marginal Gaussian statistics, nonzero-lag autocorrelation, radial PSD, or target cross-view covariance blocks inference.


In [ ]:
import pandas as pd
from IPython.display import Image as IPyImage
from IPython.display import display

diagnostics_command = [
    sys.executable,
    "-m",
    "scripts.diagnose_nile_latents",
    "--methods",
    *METHODS,
    "--seeds",
    *map(str, SEEDS),
    "--strengths",
    *map(str, STRENGTHS),
    "--azimuth-deg",
    *map(str, AZIMUTHS),
    "--frequency-scale",
    str(FREQUENCY_SCALE),
    "--camera-length-scale",
    str(CAMERA_LENGTH_SCALE),
    "--device",
    "cpu",
    "--output",
    str(DIAGNOSTICS_FILE),
]
diagnostics_returncode = run_streaming(
    diagnostics_command, cwd=COLAB_REPO, check=False
)
diagnostics_payload = json.loads(
    DIAGNOSTICS_FILE.read_text(encoding="utf-8")
)
rows = []
for record in diagnostics_payload["records"]:
    report = record.get("report", {})
    rows.append(
        {
            "method": record["method"],
            "seed": record["seed"],
            "max_correlation": record["max_correlation"],
            "passed": record.get("passed", False),
            "mean": report.get("global", {}).get("mean"),
            "std": report.get("global", {}).get("std"),
            "max_abs_lag_autocorrelation": report.get(
                "lag_autocorrelation", {}
            ).get("max_abs"),
            "radial_psd_deviation": report.get("radial_psd_deviation"),
            "covariance_mae": (
                report.get("cross_view_covariance_error") or {}
            ).get("mae"),
            "error": record.get("error"),
        }
    )
display(pd.DataFrame(rows))
assert diagnostics_returncode == 0 and diagnostics_payload["passed"], (
    f"Distribution gate failed; inspect {DIAGNOSTICS_FILE}."
)
print("All configured samplers passed the distribution gate.")


## 7. Dry-run the current isolated matrix

Only the seven distribution-preserving methods are included. The plan manifest is separate from the inference manifest. The expected smoke matrix has 11 jobs; FULL has 285 jobs.


In [ ]:
def build_grid_command(*, dry_run):
    manifest = PLAN_MANIFEST if dry_run else MANIFEST_FILE
    command = [
        sys.executable,
        "-m",
        "scripts.run_nile_grid",
        "--experiment-id",
        EXPERIMENT_ID,
        "--code-revision",
        GIT_COMMIT,
        "--input",
        *map(str, INPUT_IMAGES),
        "--methods",
        *METHODS,
        "--seeds",
        *map(str, SEEDS),
        "--strengths",
        *map(str, STRENGTHS),
        "--frequency-scale",
        str(FREQUENCY_SCALE),
        "--camera-length-scale",
        str(CAMERA_LENGTH_SCALE),
        "--text",
        TEXT_PROMPT,
        "--azimuth-deg",
        *map(str, AZIMUTHS),
        "--num-inference-steps",
        str(NUM_INFERENCE_STEPS),
        "--guidance-scale",
        str(GUIDANCE_SCALE),
        "--base-model",
        BASE_MODEL,
        "--base-model-revision",
        BASE_MODEL_REVISION,
        "--vae-model",
        VAE_MODEL,
        "--vae-model-revision",
        VAE_MODEL_REVISION,
        "--adapter-path",
        str(ADAPTER_DIR),
        "--adapter-revision",
        ADAPTER_REVISION,
        "--adapter-sha256",
        ADAPTER_SHA256,
        "--device",
        "cuda",
        "--output-root",
        str(OUTPUT_ROOT),
        "--manifest",
        str(manifest),
    ]
    if REMOVE_BACKGROUND:
        assert BIREFNET_REVISION is not None
        command.extend(
            [
                "--birefnet-model",
                BIREFNET_MODEL,
                "--birefnet-revision",
                BIREFNET_REVISION,
            ]
        )
        command.append("--remove-bg")
    command.append("--dry-run" if dry_run else "--resume")
    return command

def read_isolated_manifest(path):
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    records = payload["runs"]
    assert records, f"No records in {path}"
    assert payload.get("experiment_id") == EXPERIMENT_ID
    assert payload.get("code_revision") == GIT_COMMIT
    assert all(
        record.get("experiment_id") == EXPERIMENT_ID
        and record.get("code_revision") == GIT_COMMIT
        for record in records
    ), "Manifest contains records from another experiment/revision."
    return payload, records

run_streaming(build_grid_command(dry_run=True), cwd=COLAB_REPO)
plan_payload, plan = read_isolated_manifest(PLAN_MANIFEST)
expected_jobs = len(INPUT_IMAGES) * len(SEEDS) * (
    3 + 4 * len(STRENGTHS)
)
assert len(plan) == expected_jobs, (len(plan), expected_jobs)
assert expected_jobs == (285 if IS_FULL else 11)
print("planned jobs:", len(plan))
display(
    pd.DataFrame(plan)[
        [
            "experiment_id",
            "code_revision",
            "method",
            "seed",
            "max_correlation",
            "status",
        ]
    ]
)


## 8. Run the current isolated experiment

Resume is safe only inside this exact experiment root. The grid runner also rejects a manifest containing another experiment ID or code revision.


In [ ]:
RUN_EXPERIMENT = True
assert diagnostics_payload["passed"], "Distribution gate did not pass."
if RUN_EXPERIMENT:
    run_streaming(build_grid_command(dry_run=False), cwd=COLAB_REPO)
    print("isolated manifest:", MANIFEST_FILE)
else:
    print("Inference skipped.")


## 9. Evaluate only this experiment manifest

Lightweight metrics are collapse detectors and high-frequency guardrails, not geometry-aware consistency metrics. The evaluator receives only the isolated manifest created above.


In [ ]:
manifest_payload, manifest_records = read_isolated_manifest(MANIFEST_FILE)
status_df = pd.DataFrame(manifest_records)
display(
    status_df.groupby(["method", "status"])
    .size()
    .rename("count")
    .reset_index()
)
assert any(
    row.get("status") in {"succeeded", "skipped"}
    for row in manifest_records
)

lightweight_command = [
    sys.executable,
    "-m",
    "scripts.eval_multiview_consistency",
    "--manifest",
    str(MANIFEST_FILE),
    "--metrics",
    "lightweight",
    "--angle-bins",
    "45",
    "90",
    "135",
    "180",
    "--iid-baseline-method",
    "iid_default",
    "--image-size",
    "256",
    "--output",
    str(LIGHTWEIGHT_FILE),
]
run_streaming(lightweight_command, cwd=COLAB_REPO)
metrics = json.loads(LIGHTWEIGHT_FILE.read_text(encoding="utf-8"))
assert metrics["settings"]["lightweight_role"] == "collapse_detector_only"

angle_df = pd.DataFrame(metrics["angle_bin_summaries"])
angle_columns = [
    "experiment_id",
    "code_revision",
    "method",
    "inference_method",
    "max_correlation",
    "angle_bin_deg",
    "pair_count",
    "lowfreq_l1_similarity",
    "highfreq_l1_distance",
    "r_hf",
    "r_hf_status",
]
display(
    angle_df[
        [column for column in angle_columns if column in angle_df.columns]
    ]
)

aggregate_df = pd.DataFrame(metrics["aggregates"])
aggregate_columns = [
    "experiment_id",
    "code_revision",
    "method",
    "inference_method",
    "max_correlation",
    "camera_response_monotonic",
    "angle_all_highfreq_l1_distance",
    "r_hf",
    "r_hf_status",
]
display(
    aggregate_df[
        [
            column
            for column in aggregate_columns
            if column in aggregate_df.columns
        ]
    ]
)


In [ ]:
shown = set()
for record in manifest_records:
    method = record.get("method")
    output = Path(record["output"])
    if record.get("status") not in {"succeeded", "skipped"}:
        continue
    if method in shown or not output.is_file():
        continue
    shown.add(method)
    print(method, "strength=", record.get("max_correlation"), output)
    display(IPyImage(filename=str(output)))


## 10. Optional MEt3R

Run this only after visual inspection and collapse guardrails. For the default cosine distance, lower is better. It still reads only the current isolated manifest.


In [ ]:
INSTALL_MET3R = False
RUN_MET3R = False

if INSTALL_MET3R:
    run_pip("install", "git+https://github.com/mohammadasim98/met3r")
if RUN_MET3R:
    assert INSTALL_MET3R, "Install MEt3R first."
    command = [
        sys.executable,
        "-m",
        "scripts.eval_multiview_consistency",
        "--manifest",
        str(MANIFEST_FILE),
        "--metrics",
        "all",
        "--angle-bins",
        "45",
        "90",
        "135",
        "180",
        "--iid-baseline-method",
        "iid_default",
        "--met3r-device",
        "cuda",
        "--met3r-image-size",
        "256",
        "--met3r-batch-size",
        "1",
        "--output",
        str(MET3R_FILE),
    ]
    run_streaming(command, cwd=COLAB_REPO)
    met3r = json.loads(MET3R_FILE.read_text(encoding="utf-8"))
    assert (
        met3r["settings"]["met3r"]["score_direction"]
        == "lower_is_better"
    )
    display(pd.DataFrame(met3r["angle_bin_summaries"]))
else:
    print("MEt3R skipped; lightweight results do not replace it.")


## 11. Switching from smoke validation to the formal matrix

1. Put at least five asymmetric, user-provided images in `USER_INPUT_DIR`.
2. Set `RUN_PROFILE = "FULL"`.
3. Run all cells again. FULL hard-asserts exactly five selected inputs, seeds `0/1/2`, and strengths `0.15/0.30/0.45/0.60`.
4. The new profile and complete configuration hash create a different experiment root and manifest automatically.

Never report `QUICK_SMOKE` output as a formal experiment. Do not re-enable Sobol, the old low-pass sampler, or denoising callbacks here.
